In [1]:
from datetime import date

today = date.today()
print(f"Today's date is {today}. its me Sahana.D !")

Today's date is 2025-03-05. its me Sahana.D !


In [2]:
from google.colab import drive

# This will prompt you to authorize access
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#datacamp project
Pneumonia is one of the leading respiratory illnesses worldwide, and its timely and accurate diagnosis is essential for effective treatment. Manually reviewing chest X-rays is a critical step in this process, and AI can provide valuable support by helping to expedite the assessment. In your role as a consultant data scientist, you will test the ability of a deep learning model to distinguish pneumonia cases from normal images of lungs in chest X-rays.

By fine-tuning a pre-trained convolutional neural network, specifically the ResNet-18 model, your task is to classify X-ray images into two categories: normal lungs and those affected by pneumonia. You can leverage its already trained weights and get an accurate classifier trained faster and with fewer resources.

## The Data

<img src="x-rays_sample.png" align="center"/>
&nbsp

You have a dataset of chest X-rays that have been preprocessed for use with a ResNet-18 model. You can see a sample of 5 images from each category above. Upon unzipping the `chestxrays.zip` file (code provided below), you will find your dataset inside the `data/chestxrays` folder divided into `test` and `train` folders.

There are 150 training images and 50 testing images for each category, NORMAL and PNEUMONIA (300 and 100 in total). For your convenience, this data has already been loaded into a `train_loader` and a `test_loader` using the `DataLoader` class from the PyTorch library.

In [3]:
#pip install torch torchvision


In [4]:
#!pip install torchmetrics

In [5]:
import torch
import torchvision
print(torch.__version__)
print(torchvision.__version__)


2.5.1+cu124
0.20.1+cu124


In [6]:
# Import required libraries
# -------------------------
# Data loading
import random
import numpy as np
from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Train model
import torch
from torchvision import models
import torch.nn as nn
import torch.optim as optim

# Evaluate model
from torchmetrics import Accuracy, F1Score

# Set random seeds for reproducibility
torch.manual_seed(101010)
np.random.seed(101010)
random.seed(101010)

In [7]:
import os
import zipfile

# Unzip the data folder
if not os.path.exists('/content/drive/MyDrive/ImageClassification-Pneumonia/chestxrays'):
    with zipfile.ZipFile('/content/drive/MyDrive/ImageClassification-Pneumonia/chestxrays/chestxrays.zip', 'r') as zip_ref:
        zip_ref.extractall('data')

In [8]:
# Define the transformations to apply to the images for use with ResNet-18
transform_mean = [0.485, 0.456, 0.406]
transform_std =[0.229, 0.224, 0.225]
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize(mean=transform_mean, std=transform_std)])

# Apply the image transforms
train_dataset = ImageFolder('/content/drive/MyDrive/ImageClassification-Pneumonia/chestxrays/train', transform=transform)
test_dataset = ImageFolder('/content/drive/MyDrive/ImageClassification-Pneumonia/chestxrays/test', transform=transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=len(train_dataset) // 2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset))

In [9]:
# Start coding here
# Use as many cells as you need
# Load the pre-trained ResNet-18 model
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)



In [10]:
# Modify the model

# Freeze the parameters of the model
for param in resnet18.parameters():
    param.requires_grad = False

# Modify the final layer for binary classification
resnet18.fc = nn.Linear(resnet18.fc.in_features, 1)


In [11]:
#  Define the training loop


# Model training/fine-tuning loop
def train(model, train_loader, criterion, optimizer, num_epochs):

    # Train the model for the specified number of epochs
    for epoch in range(num_epochs):
        # Set the model to train mode
        model.train()

        # Initialize the running loss and accuracy
        running_loss = 0.0
        running_accuracy = 0

        # Iterate over the batches of the train loader
        for inputs, labels in train_loader:

            # Zero the optimizer gradients
            optimizer.zero_grad()

            # Ensure labels have the same dimensions as outputs
            labels = labels.float().unsqueeze(1)

            # Forward pass
            outputs = model(inputs)
            preds = torch.sigmoid(outputs) > 0.5 # Binary classification
            loss = criterion(outputs, labels)

            # Backward pass and optimizer step
            loss.backward()
            optimizer.step()

            # Update the running loss and accuracy
            running_loss += loss.item() * inputs.size(0)
            running_accuracy += torch.sum(preds == labels.data)

        # Calculate the train loss and accuracy for the current epoch
        train_loss = running_loss / len(train_dataset)
        train_acc = running_accuracy.double() / len(train_dataset)

        # Print the epoch results
        print('Epoch [{}/{}], train loss: {:.4f}, train acc: {:.4f}'
              .format(epoch+1, num_epochs, train_loss, train_acc))



In [16]:
#-------------------------
#Fine-tune the model

# Set the model to ResNet-18
model = resnet18

# Fine-tune the ResNet-18 model for 3 epochs using the train_loader//3 to 20
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.01)
criterion = torch.nn.BCEWithLogitsLoss()
train(model, train_loader, criterion, optimizer, num_epochs=20)



Epoch [1/20], train loss: 0.5961, train acc: 0.7567
Epoch [2/20], train loss: 0.2853, train acc: 0.8833
Epoch [3/20], train loss: 0.5284, train acc: 0.7867
Epoch [4/20], train loss: 0.2471, train acc: 0.9233
Epoch [5/20], train loss: 0.3179, train acc: 0.8467
Epoch [6/20], train loss: 0.3046, train acc: 0.8600
Epoch [7/20], train loss: 0.2111, train acc: 0.9233
Epoch [8/20], train loss: 0.2648, train acc: 0.9033
Epoch [9/20], train loss: 0.2546, train acc: 0.9100
Epoch [10/20], train loss: 0.1870, train acc: 0.9167
Epoch [11/20], train loss: 0.1996, train acc: 0.9100
Epoch [12/20], train loss: 0.2069, train acc: 0.9100
Epoch [13/20], train loss: 0.1718, train acc: 0.9300
Epoch [14/20], train loss: 0.1895, train acc: 0.9267
Epoch [15/20], train loss: 0.1804, train acc: 0.9367
Epoch [16/20], train loss: 0.1900, train acc: 0.9300
Epoch [17/20], train loss: 0.1526, train acc: 0.9333
Epoch [18/20], train loss: 0.1566, train acc: 0.9367
Epoch [19/20], train loss: 0.1431, train acc: 0.9467
Ep

evaluate the accuracy and F1-score of your fine-tuned model.

In [20]:
#-------------------
# Evaluate the model
#-------------------

# Set model to evaluation mode
model = resnet18
model.eval()

# Initialize metrics for accuracy and F1 score
accuracy_metric = Accuracy(task="binary")
f1_metric = F1Score(task="binary")

# Create lists store all predictions and labels
all_preds = []
all_labels = []

# Disable gradient calculation for evaluation
with torch.no_grad():
  for inputs, labels in test_loader:
    # Forward pass
    outputs = model(inputs)
    preds = torch.sigmoid(outputs).round()  # Round to 0 or 1

    # Extend the lists with predictions and labels
    all_preds.extend(preds.tolist())
    all_labels.extend(labels.unsqueeze(1).tolist())

    # Convert lists back to tensors
    all_preds = torch.tensor(all_preds)
    all_labels = torch.tensor(all_labels)

    # Calculate accuracy and F1 score
    test_acc = accuracy_metric(all_preds, all_labels).item()
    test_f1 = f1_metric(all_preds, all_labels).item()


print(f"\nTest accuracy: {test_acc:.3f}\nTest F1-score: {test_f1:.3f}")



Test accuracy: 0.740
Test F1-score: 0.794


Conclusion ⁉
Model Performance

   if epoch =3  Test accuracy: 0.500 (50%)  Test F1-score: 0.667 (66.7%)

   if epoch =10 Test accuracy: 0.800 (80%) Test F1-score: 0.815(81.5 %)

   if epoch =20 Test accuracy: 0.74 (74%) Test F1-score: 0.794(79.4 %)

Need to look further. Have to change the fine tune strategy